# Montgom Project
Use the Python (MTGP) kernel after completing the setup in the project README.

In [ ]:
import mtgp

print(mtgp.__file__)

In [ ]:
from mtgp import Trial
import matplotlib.pyplot as plt
import numpy as np

cmap = plt.colormaps["gray"].with_extremes(bad="red")

trial = Trial()
trial.data

In [ ]:
patient = trial[0]
print(patient)
print(patient.clinical_reading)
print("Shape:", patient.matrix_size, "Spacing (mm):", patient.pixel_spacing_mm)
patient.metadata

In [ ]:
import matplotlib.pyplot as plt

for patient in trial[:5]:
    image = patient.image
    left, right = patient.left_mask, patient.right_mask
    fig, axes = plt.subplots(1, 3, figsize=(15, 6))
    for ax, pixels, title in zip(axes, [image, left, right], ["X-ray", "Left lung", "Right lung"]):
        ax.imshow(pixels, cmap="gray")
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8), layout="constrained")
for ax, patient in zip(axes.flat, trial[:6]):
    ax.imshow(patient.image, cmap="gray")
    ax.set_title(patient.patient_id)

plt.show()

Mean image threshold images

In [ ]:
import numpy as np
cmap = plt.colormaps["gray"].with_extremes(bad="red")

fig2, axs2 = plt.subplots(2, 3)

for i, patient in enumerate(trial[:6]):
    image = np.array(patient.image, dtype=np.float32)
    valid = image >= 2
    image[~valid] = np.nan

    thresh = np.nanmean(image)
    image_t = np.where(valid, image < thresh, np.nan)

    axs2[i // 3, i % 3].imshow(image_t, cmap=cmap, vmin=0, vmax=1)
    
plt.show()


## Separate boundary strips and repair lung blobs

The pipeline is implemented in `mtgp.thresholding` and `mtgp.morphology`; plotting
lives in `mtgp.plotting`. Images are processed at their original resolution.

1. Smooth only valid intensities using normalized Gaussian weights.
2. Threshold the smoothed image.
3. **Open the white foreground** to cut thin connections to boundary strips
   (equivalently, close narrow gaps in the black background).
4. Relabel the separated shapes, close each individually, and fill enclosed holes.
5. Reject shapes touching the image edge or too many remaining invalid pixels.

Red means invalid image data. Enclosed invalid holes become white when filled,
but invalid areas connected to the outside are not used as bridges. The method
allows several separate lung fragments; it does not force two components.

Opening can round lung tips, remove small fragments, and leave detached border
remnants. Wider connections can survive. Review the intermediate masks before
interpreting these threshold masks as anatomical segmentation.

In [ ]:
from mtgp import SegmentationConfig, segment_lungs
from mtgp.plotting import plot_segmentation_batch

config = SegmentationConfig(
    sigma=30,
    threshold=65,
    min_valid_intensity=10,
    opening_radius=100,       # Pixels: increase to cut wider white connections.
    closing_radius=100,       # Pixels: close black clefts inside each blob.
    max_invalid_contacts=100,
)

### Inspect the first 30 patients

Each figure has patient IDs and only as many rows as required. The threshold
line uses `config.threshold`. Plot previews are reduced for display to avoid
retaining dozens of full-resolution arrays; segmentation and histograms still
use every pixel. For a quick experiment, change the selection to `trial[:3]`.
The attached problem examples are `trial["MCUCXR_0022_0"]` and
`trial["MCUCXR_0042_0"]`; pass them as a list to inspect just those two images.

Set `opening_radius=0` to compare with the previous individual-closing pipeline.
Set `closing_radius=0` to disable closing (enclosed holes are still filled).
All distances are in input pixels, so adjust them if you change image resolution.

In [ ]:
figures = plot_segmentation_batch(trial[:30], config)
plt.show()

### Use a mask directly

`segment_lungs` returns the boolean final mask plus all intermediate arrays.
The original validity mask is retained separately from `mask_valid`, which also
admits invalid holes filled during repair. No patient data is modified.

In [ ]:
patient = trial[0]
result = segment_lungs(patient.image, config)
lung_mask = result.cleaned

fig, ax = plt.subplots(figsize=(7, 6), layout="constrained")
ax.imshow(
    np.where(result.mask_valid, lung_mask.astype(float), np.nan),
    cmap=cmap, vmin=0, vmax=1,
)
ax.set_title(f"{patient.patient_id}: repaired lung mask")
plt.show()